# Mejorando YOLO con SAHI — Demo interactiva
## Detección y segmentación de objetos pequeños en vista aérea urbana
### FLISOL 2026 · Computer Vision Open Source

Recorrido:

1. **El problema** — por qué YOLO falla con objetos pequeños (+ estadísticas del dataset).
2. **SAHI** — slicing, y cómo se **reparten los slices** con/sin overlap.
3. **Parámetros** — tamaño de slice, overlap, postprocesamiento (NMS/NMM/GREEDYNMM), modos.
4. **¿Es consistente?** — la mejora sobre **varias imágenes**, no una sola.
5. **Segmentación** — YOLO-seg + SAHI, con los **mejores casos**.

Dataset: **VisDrone2019-DET**.  ⚙️ Activa la **GPU (T4)**.

---
## 1. Instalación y entorno

In [ ]:
!pip install -q ultralytics sahi supervision matplotlib
import torch, time, glob, os
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.is_available(), '| device:', DEVICE)

---
## 2. El problema, en una imagen

YOLO **redimensiona toda la imagen** a su tamaño de entrada (típicamente 640×640).

| Imagen original | Tras redimensionar a 640 | Efecto |
|---|---|---|
| 1920×1080 | 640×360 (escala 0.33) | una persona de 30px → 10px |
| 3840×2160 (4K) | 640×360 (escala 0.17) | una moto de 40px → 7px |

A 7-10 px el objeto pierde casi toda su información y el detector **no lo ve**.

### 2.1 Descargar VisDrone (val)

In [ ]:
import urllib.request, zipfile
URL = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/VisDrone2019-DET-val.zip'
if not os.path.exists('VisDrone2019-DET-val'):
    print('Descargando VisDrone val (~80 MB)...')
    urllib.request.urlretrieve(URL, 'visdrone_val.zip')
    with zipfile.ZipFile('visdrone_val.zip') as z: z.extractall('.')
    print('Listo.')
imgs = sorted(glob.glob('VisDrone2019-DET-val/images/*.jpg'))
print('Imágenes disponibles:', len(imgs))
from PIL import Image
IMG = imgs[3]
im = Image.open(IMG); print('Imagen:', IMG, '| resolución:', im.size); im

### 2.2 ¿Cuán pequeños son los objetos? (estadísticas del dataset)

Antes de detectar nada, midamos el dataset. El criterio COCO llama **'small'** a un objeto con
√área < 32 px. Veamos qué fracción de VisDrone cae ahí — esto justifica usar SAHI.

In [ ]:
import math
import matplotlib.pyplot as plt
sides, per_img = [], []
for ann in glob.glob('VisDrone2019-DET-val/annotations/*.txt'):
    n = 0
    for line in open(ann):
        p = line.strip().rstrip(',').split(',')
        if len(p) < 6: continue
        w, h, cat = float(p[2]), float(p[3]), int(p[5])
        if cat in (0, 11) or w <= 0 or h <= 0: continue
        sides.append(math.sqrt(w*h)); n += 1
    per_img.append(n)
small = sum(s < 32 for s in sides); total = len(sides)
print(f'{total} objetos en {len(per_img)} imágenes | promedio {sum(per_img)/len(per_img):.0f} obj/img')
print(f'{100*small/total:.1f}% de los objetos son SMALL (<32 px)')

fig, ax = plt.subplots(1, 2, figsize=(16, 5))
ax[0].hist([s for s in sides if s < 200], bins=60, color='#3A6EA5')
ax[0].axvline(32, color='#E4572E', ls='--', lw=2, label='32 px (small)')
ax[0].axvline(96, color='#444', ls='--', lw=2, label='96 px (large)')
ax[0].set_title('Distribución de tamaños de objeto'); ax[0].set_xlabel('√área (px)'); ax[0].legend()
ax[1].hist(per_img, bins=40, color='#2E8B57')
ax[1].set_title('Objetos por imagen (densidad)'); ax[1].set_xlabel('objetos/imagen')
plt.show()

---
## 3. YOLO tradicional (el *antes*)

Usamos `yolo11s.pt` (COCO). Fíjate cuántos objetos lejanos se **quedan sin caja**.

In [ ]:
from ultralytics import YOLO
import cv2
model = YOLO('yolo11s.pt')
t0 = time.perf_counter()
res = model.predict(IMG, conf=0.25, device=DEVICE, verbose=False)[0]
dt_base = time.perf_counter() - t0; n_base = len(res.boxes)
print(f'YOLO baseline: {n_base} detecciones en {dt_base:.3f}s')
cv2.imwrite('baseline.jpg', res.plot())   # res.plot() -> BGR, funciona en toda version
Image.open('baseline.jpg')

---
## 4. YOLO + SAHI (el *después*)

SAHI **envuelve** a YOLO: corta en slices con solape, detecta en cada uno, reproyecta y **fusiona**.

In [ ]:
from sahi import AutoDetectionModel
from sahi.predict import get_prediction, get_sliced_prediction
sahi_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics', model_path='yolo11s.pt', confidence_threshold=0.25, device=DEVICE)
t0 = time.perf_counter()
result = get_sliced_prediction(IMG, sahi_model, slice_height=640, slice_width=640,
    overlap_height_ratio=0.2, overlap_width_ratio=0.2,
    postprocess_type='GREEDYNMM', postprocess_match_metric='IOS',
    postprocess_match_threshold=0.5, verbose=0)
dt_sahi = time.perf_counter() - t0; n_sahi = len(result.object_prediction_list)
print(f'YOLO + SAHI: {n_sahi} detecciones en {dt_sahi:.3f}s  ({dt_sahi/max(dt_base,1e-6):.1f}x mas lento)')
result.export_visuals(export_dir='.', file_name='sahi'); Image.open('sahi.png')

### 4.1 Comparativa lado a lado

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(22, 9))
ax[0].imshow(Image.open('baseline.jpg')); ax[0].set_title(f'YOLO  -  {n_base} objetos', fontsize=18); ax[0].axis('off')
ax[1].imshow(Image.open('sahi.png'));     ax[1].set_title(f'YOLO + SAHI  -  {n_sahi} objetos', fontsize=18); ax[1].axis('off')
plt.tight_layout(); plt.show()
print(f'SAHI encontro {n_sahi - n_base} objetos adicionales (x{n_sahi/max(n_base,1):.1f}).')

### 4.2 ¿Cómo reparte SAHI los slices? (overlap visual)

Dibujamos la rejilla de slices. Donde dos slices se superponen, el color se acumula y se ve **más oscuro**.
Sin overlap, un objeto en el borde se parte; con overlap, aparece completo en algún slice.

In [ ]:
import matplotlib.patches as mpatches
from sahi.slicing import get_slice_bboxes
import numpy as np
img_rgb = np.array(Image.open(IMG)); H, W = img_rgb.shape[:2]
fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, ov in zip(axes, [0.0, 0.2, 0.4]):
    boxes = get_slice_bboxes(H, W, slice_height=400, slice_width=400,
                             overlap_height_ratio=ov, overlap_width_ratio=ov)
    ax.imshow(img_rgb)
    for (x1, y1, x2, y2) in boxes:
        ax.add_patch(mpatches.Rectangle((x1, y1), x2-x1, y2-y1, facecolor='#2E8B57',
                     alpha=0.18, edgecolor='white', lw=1.2))
    ax.set_title(('Sin solape' if ov == 0 else f'Overlap {ov}') + f'  ·  {len(boxes)} slices', fontsize=15)
    ax.axis('off')
plt.tight_layout(); plt.show()

---
## 5. Parámetro 1 — Tamaño de slice (trade-off precisión/velocidad)

In [ ]:
sizes, dets, times = [320, 512, 640, 768, 1024], [], []
for s in sizes:
    t0 = time.perf_counter()
    r = get_sliced_prediction(IMG, sahi_model, slice_height=s, slice_width=s,
                              overlap_height_ratio=0.2, overlap_width_ratio=0.2,
                              postprocess_type='GREEDYNMM', verbose=0)
    dt = time.perf_counter()-t0; dets.append(len(r.object_prediction_list)); times.append(dt)
    print(f'slice={s:4d}px -> {len(r.object_prediction_list):4d} det en {dt:.2f}s')
fig, a1 = plt.subplots(figsize=(10, 5))
a1.bar([str(s) for s in sizes], dets, color='#2E8B57', alpha=0.85); a1.set_ylabel('Detecciones', color='#2E8B57')
a2 = a1.twinx(); a2.plot([str(s) for s in sizes], times, 'o-', color='#E4572E', lw=2.5); a2.set_ylabel('Tiempo (s)', color='#E4572E')
a1.set_xlabel('Tamano de slice (px)'); plt.title('Precision vs velocidad'); plt.show()

---
## 6. Parámetro 2 — Overlap

In [ ]:
for ov in [0.0, 0.1, 0.2, 0.3]:
    t0 = time.perf_counter()
    r = get_sliced_prediction(IMG, sahi_model, slice_height=512, slice_width=512,
                              overlap_height_ratio=ov, overlap_width_ratio=ov,
                              postprocess_type='GREEDYNMM', verbose=0)
    print(f'overlap={ov:.1f} -> {len(r.object_prediction_list):4d} det en {time.perf_counter()-t0:.2f}s')

---
## 7. Parámetro 3 — Postprocesamiento (fusión de duplicados)

| Tipo | Qué hace |
|---|---|
| **NMS** | descarta la caja de menor score cuando dos se solapan |
| **NMM** | fusiona las cajas solapadas en una sola |
| **GREEDYNMM** | variante voraz de NMM (default de SAHI) |

Métrica de match: `IOU` (solape clásico) o `IOS` (Intersection over Smaller).

In [ ]:
for pp in ['NMS', 'NMM', 'GREEDYNMM']:
    t0 = time.perf_counter()
    r = get_sliced_prediction(IMG, sahi_model, slice_height=640, slice_width=640,
                              overlap_height_ratio=0.2, overlap_width_ratio=0.2,
                              postprocess_type=pp, postprocess_match_metric='IOS',
                              postprocess_match_threshold=0.5, verbose=0)
    print(f'{pp:10s} -> {len(r.object_prediction_list):4d} det tras fusion en {time.perf_counter()-t0:.2f}s')

---
## 8. Modos de inferencia

- **Sliced**: solo slices → objetos pequeños.
- **Standard**: imagen completa → objetos grandes.
- **Combinado** (`perform_standard_pred=True`): ambos fusionados → lo más robusto.

In [ ]:
rss = get_sliced_prediction(IMG, sahi_model, slice_height=640, slice_width=640,
        overlap_height_ratio=0.2, overlap_width_ratio=0.2, perform_standard_pred=False,
        postprocess_type='GREEDYNMM', verbose=0)
rco = get_sliced_prediction(IMG, sahi_model, slice_height=640, slice_width=640,
        overlap_height_ratio=0.2, overlap_width_ratio=0.2, perform_standard_pred=True,
        postprocess_type='GREEDYNMM', verbose=0)
print('Solo sliced:', len(rss.object_prediction_list), '| Combinado:', len(rco.object_prediction_list))

---
## 9. ¿Es consistente la mejora? — varias imágenes

Una sola imagen puede ser anecdótica. Repetimos sobre **8 imágenes** y comparamos el conteo YOLO vs SAHI.

In [ ]:
sample = imgs[:8]
nb, ns = [], []
for p in sample:
    nb.append(len(model.predict(p, conf=0.25, device=DEVICE, verbose=False)[0].boxes))
    r = get_sliced_prediction(p, sahi_model, slice_height=640, slice_width=640,
                              overlap_height_ratio=0.2, overlap_width_ratio=0.2,
                              postprocess_type='GREEDYNMM', verbose=0)
    ns.append(len(r.object_prediction_list))
x = range(len(sample)); w = 0.4
fig, ax = plt.subplots(figsize=(12, 5))
ax.bar([i-w/2 for i in x], nb, w, label='YOLO', color='#E4572E')
ax.bar([i+w/2 for i in x], ns, w, label='YOLO+SAHI', color='#2E8B57')
ax.set_xticks(list(x)); ax.set_xticklabels([f'img {i+1}' for i in x])
ax.set_ylabel('Detecciones'); ax.set_title('Mejora consistente en 8 imágenes'); ax.legend()
plt.show()
print(f'Promedio YOLO: {sum(nb)/len(nb):.0f} | YOLO+SAHI: {sum(ns)/len(ns):.0f} | x{sum(ns)/max(sum(nb),1):.1f}')

---
## 10. SAHI con SEGMENTACIÓN de instancias

SAHI también funciona con modelos `-seg` (máscaras). Cada slice produce máscaras a resolución nativa
y se fusionan, así que las instancias pequeñas obtienen máscara — o aparecen cuando antes no estaban.

> VisDrone no trae máscaras de ground truth → esta sección es **cualitativa**.

In [ ]:
seg_model = AutoDetectionModel.from_pretrained(
    model_type='ultralytics', model_path='yolo11s-seg.pt', confidence_threshold=0.25, device=DEVICE)

def seg_panel(path, name):
    a = get_prediction(path, seg_model); a.export_visuals(export_dir='.', file_name=name+'_std')
    b = get_sliced_prediction(path, seg_model, slice_height=640, slice_width=640,
                              overlap_height_ratio=0.2, overlap_width_ratio=0.2,
                              postprocess_type='GREEDYNMM', verbose=0)
    b.export_visuals(export_dir='.', file_name=name+'_sahi')
    return len(a.object_prediction_list), len(b.object_prediction_list)

ns_std, ns_sahi = seg_panel(IMG, 'seg')
print(f'Segmentacion -> YOLO-seg: {ns_std} | YOLO-seg+SAHI: {ns_sahi} mascaras')
fig, ax = plt.subplots(1, 2, figsize=(22, 9))
ax[0].imshow(Image.open('seg_std.png'));  ax[0].set_title(f'YOLO-seg  -  {ns_std}', fontsize=18); ax[0].axis('off')
ax[1].imshow(Image.open('seg_sahi.png')); ax[1].set_title(f'YOLO-seg + SAHI  -  {ns_sahi}', fontsize=18); ax[1].axis('off')
plt.tight_layout(); plt.show()

### 10.1 Mejores casos de segmentación (varias imágenes)

Probamos varias imágenes y mostramos las **2 con mayor ganancia** de instancias segmentadas.

In [ ]:
cand = imgs[4:14]
gains = []
for i, p in enumerate(cand):
    a, b = seg_panel(p, f'segc{i}')
    gains.append((b - a, a, b, i, p))
gains.sort(reverse=True)
for gain, a, b, i, p in gains[:2]:
    fig, ax = plt.subplots(1, 2, figsize=(22, 9))
    ax[0].imshow(Image.open(f'segc{i}_std.png'));  ax[0].set_title(f'YOLO-seg  -  {a}', fontsize=16); ax[0].axis('off')
    ax[1].imshow(Image.open(f'segc{i}_sahi.png')); ax[1].set_title(f'YOLO-seg + SAHI  -  {b}  (+{gain})', fontsize=16); ax[1].axis('off')
    plt.tight_layout(); plt.show()

---
## 11. ¿Y los números (mAP)?

Esta demo es **cualitativa**. Para el **benchmark de mAP** (incl. `AP_small`, AP por clase, postprocesamiento)
usa **`02_resultados_figuras.ipynb`**, que evalúa contra el ground truth con `pycocotools`.

> El mAP real requiere el modelo **entrenado en VisDrone** (`00_entrenamiento.ipynb`).

---
## Conclusiones

- VisDrone es mayoritariamente **objetos pequeños** → YOLO los pierde al aplastar la imagen.
- SAHI **rebana** y los recupera, de forma **consistente** entre imágenes, **sin reentrenar**.
- Controlas: **tamaño de slice, overlap, modo y postprocesamiento**.
- Funciona para **detección y segmentación**.
- Costo: **tiempo** (~proporcional al nº de slices). 100% **open source**.

Repo: github.com/jossuema/yolo-sahi-flisol2026